# Survival model - Charlson

In [ ]:
import yaml
import sys
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
import numpy as np
from lifelines import CoxPHFitter

MODEL_CONFIG = yaml.safe_load(open("../config.yml"))["models"]["charlson_1_year_survival"]

## Index-Specific Train/Test Splits

Create views of the train/test splits which have the comorbidities for each index present.

### Data Aggregation

The training data can be quite large which results in extremely large inputs for the models. For example, the MACSS has 100 comorbidities so the training data is a N x 100 matrix where N is the number of rows.

To improve the training time for our models, we can 'compress' the data and represent it by identifying each unique combination of comorbidities and the number of times it occurred.

For example, given the following row-level data:

| COMORB_1 | COMORB_2 | COMORB_3 |
| -------- | -------- | -------- |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    0     |
|    1     |    0     |    1     |
|    1     |    0     |    1     |

We would represent it as:

| COMORB_1 | COMORB_2 | COMORB_3 | N |
| -------- | -------- | -------- | - |
|    1     |    1     |    1     | 7 |
|    1     |    1     |    0     | 1 |
|    1     |    0     |    1     | 2 |

In [ ]:
"""
This imports the data from csv files. The code can be modified to read from a different source
if preferred, such as a database or an API.
"""

charls_survival_training = pd.read_csv(f"../datasets/{MODEL_CONFIG["dataset_training"]}")

In [ ]:
# Check the first 10 rows of the training data
charls_survival_training.head(10)

In [ ]:
charls_survival_training.columns

In [ ]:
# Group by the training and testing data to get the sample weight 
charls_survival_training_agg = charls_survival_training.groupby(list(charls_survival_training.columns),dropna=False).size().reset_index(name='N')

In [ ]:
# Reducation in training rows
print(f"Reduction in training rows: {charls_survival_training.shape[0] - charls_survival_training_agg.shape[0]}")

In [ ]:
# Check the class
MODEL_CONFIG["indicator"]

In [ ]:
# Define the input columns for the model
input_cols = [i for i in charls_survival_training_agg.columns if "C_" in i]

In [ ]:
include_cols = input_cols + ['ONE_YEAR_MORT', 'SURVIVAL_TIME_DAYS', 'N']

In [ ]:
# Fit the Cox proportional hazards model
# We add a penalizer to handle potential collinearity or convergence issues
cph = CoxPHFitter(penalizer=0.1)

# Prepare data for fitting
data_to_fit = charls_survival_training_agg[include_cols].copy()

# Drop any rows with NaNs
data_to_fit = data_to_fit.dropna()

# Ensure survival time is positive (replace 0 with a small epsilon)
data_to_fit['SURVIVAL_TIME_DAYS'] = data_to_fit['SURVIVAL_TIME_DAYS'].replace(0, 0.1)

# Fit the model
cph.fit(data_to_fit, duration_col='SURVIVAL_TIME_DAYS', event_col='ONE_YEAR_MORT', weights_col='N')

# Print summary
print(cph.summary)

In [ ]:
""" 
Calculating the scores for the Charlson Comorbidity Index
Based on the hazard ratio as described in the 
following papers:

Hude Quan, Bing Li, Chantal M. Couris, Kiyohide Fushimi, Patrick Graham, Phil Hider, Jean-Marie Januel, 
Vijaya Sundararajan, Updating and Validating the Charlson Comorbidity Index and Score for Risk Adjustment 
in Hospital Discharge Abstracts Using Data From 6 Countries 
https://doi.org/10.1093/aje/kwq433

Mary E. Charlson, Peter Pompei, Kathy L. Ales, C.Ronald MacKenzie,
A new method of classifying prognostic comorbidity in longitudinal studies: Development and validation,
https://doi.org/10.1016/0021-9681(87)90171-8.

"""
variable_list = []
score_list = [] 
for i in range(len(cph.hazard_ratios_)):
    variable = cph.hazard_ratios_.index[i]
    score = cph.hazard_ratios_[variable]
    variable_list.append(variable)
    if score < 1.2:
        score_list.append(0)
    elif score >= 1.2 and score < 1.5:
        score_list.append(1)
    elif score >= 1.5 and score < 2.5:
        score_list.append(2)
    elif score >= 2.5 and score < 3.5:
        score_list.append(3)
    elif score >= 3.5 and score < 4.5:
        score_list.append(4)
    elif score >= 4.5 and score < 6:
        score_list.append(5)
    else:
        score_list.append(6)

In [ ]:
for score in zip(variable_list, score_list):
    print(f"{score[0]}: {score[1]}")

In [ ]:
# Create a DataFrame with these new scores
# and the original 1992 scores (data from 1984)
# and the updated scores from 2012 (Note the data was from 2004 and 2008)

charls_scores = pd.DataFrame({
    "variable": variable_list,
    "score_1992": [1,1,1,1,1,1,1,1,1,1,2,2,2,2,3,6,6],
    "score_2012": [0,2,0,0,2,1,1,0,2,0,1,2,1,2,4,6,4],
    "new_scores": score_list
})

charls_scores.head(20)